<a href="https://colab.research.google.com/github/asetya/BigData/blob/master/vision_transformer_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vision Transformer (ViT) from scratch — PyTorch

This notebook implements a Vision Transformer from scratch, matching the patch embedding, self-attention, and encoder block math from the earlier walkthrough, then trains it on CIFAR-10.

**Runtime:** in Colab, go to `Runtime > Change runtime type > GPU` (T4 is fine) before running.

**Contents**
1. Setup
2. Patch embedding
3. Multi-head self-attention
4. Transformer encoder block
5. Full Vision Transformer model
6. Shape walkthrough (sanity check)
7. Training on CIFAR-10
8. Visualizing attention on a real image
9. Scaling up to ViT-Base / next steps

## 1. Setup

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## 2. Patch embedding

Split the image into fixed-size patches, flatten each one, and linearly project it into a `D`-dimensional token — exactly the `z0 = x_p E` step from the walkthrough. As noted there, this is implemented as a single `Conv2d` with `kernel_size = stride = patch_size`, which is mathematically identical to "chop into non-overlapping patches, apply the same dense layer to each."

In [2]:
class PatchEmbedding(nn.Module):
    """Splits an image into patches and linearly projects each into embed_dim."""
    def __init__(self, img_size=32, patch_size=4, in_channels=3, embed_dim=192):
        super().__init__()
        assert img_size % patch_size == 0, "image size must be divisible by patch size"
        self.num_patches = (img_size // patch_size) ** 2
        # Conv with kernel=stride=patch_size == "flatten patch + linear projection E"
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        # x: (B, C, H, W)
        x = self.proj(x)                  # (B, embed_dim, H/P, W/P)
        x = x.flatten(2)                  # (B, embed_dim, N)
        x = x.transpose(1, 2)             # (B, N, embed_dim)  <- sequence of patch tokens
        return x

In [3]:
# Quick check: 32x32 image, patch size 4 -> (32/4)^2 = 64 patches
pe = PatchEmbedding(img_size=32, patch_size=4, embed_dim=192)
dummy_img = torch.randn(2, 3, 32, 32)
tokens = pe(dummy_img)
print("Patch tokens shape:", tokens.shape)  # (2, 64, 192)

Patch tokens shape: torch.Size([2, 64, 192])


## 3. Multi-head self-attention

This is the exact formula from the walkthrough:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

implemented with `h` parallel heads, each with its own learned `Wq`, `Wk`, `Wv`, concatenated and projected through `Wo` at the end.

In [4]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim=192, num_heads=3, attn_dropout=0.0, proj_dropout=0.0):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads          # d_k per head
        self.scale = self.head_dim ** -0.5               # 1 / sqrt(d_k)

        self.qkv = nn.Linear(embed_dim, embed_dim * 3)   # produces Q, K, V in one matmul
        self.attn_dropout = nn.Dropout(attn_dropout)
        self.proj = nn.Linear(embed_dim, embed_dim)      # W_O
        self.proj_dropout = nn.Dropout(proj_dropout)

    def forward(self, x, return_attn=False):
        B, N, D = x.shape
        qkv = self.qkv(x)                                          # (B, N, 3D)
        qkv = qkv.reshape(B, N, 3, self.num_heads, self.head_dim)   # (B, N, 3, h, d_k)
        qkv = qkv.permute(2, 0, 3, 1, 4)                            # (3, B, h, N, d_k)
        q, k, v = qkv[0], qkv[1], qkv[2]                            # each: (B, h, N, d_k)

        attn_scores = (q @ k.transpose(-2, -1)) * self.scale        # QK^T / sqrt(d_k)  -> (B, h, N, N)
        attn_weights = attn_scores.softmax(dim=-1)                  # softmax over keys
        attn_weights = self.attn_dropout(attn_weights)

        out = attn_weights @ v                                      # weighted sum of V -> (B, h, N, d_k)
        out = out.transpose(1, 2).reshape(B, N, D)                  # concat heads -> (B, N, D)
        out = self.proj_dropout(self.proj(out))                    # final W_O projection

        if return_attn:
            return out, attn_weights
        return out

In [5]:
# Quick check + peek at the attention weights matrix
mhsa = MultiHeadSelfAttention(embed_dim=192, num_heads=3)
out, attn_w = mhsa(tokens, return_attn=True)
print("Output shape:", out.shape)          # (2, 64, 192) -- same shape as input, as expected
print("Attention weights shape:", attn_w.shape)  # (2, heads, N, N) -- one NxN matrix per head, per image
print("Each row sums to 1 (softmax):", attn_w[0, 0, 0].sum().item())

Output shape: torch.Size([2, 64, 192])
Attention weights shape: torch.Size([2, 3, 64, 64])
Each row sums to 1 (softmax): 1.0


## 4. Transformer encoder block

Wraps attention and an MLP with pre-norm and residual connections:

$$z'_\ell = \text{MHA}(\text{LN}(z_{\ell-1})) + z_{\ell-1}$$
$$z_\ell = \text{MLP}(\text{LN}(z'_\ell)) + z'_\ell$$

In [6]:
class MLP(nn.Module):
    def __init__(self, embed_dim, hidden_dim, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class EncoderBlock(nn.Module):
    def __init__(self, embed_dim=192, num_heads=3, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadSelfAttention(embed_dim, num_heads, attn_dropout=dropout, proj_dropout=dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = MLP(embed_dim, int(embed_dim * mlp_ratio), dropout=dropout)

    def forward(self, x, return_attn=False):
        if return_attn:
            attn_out, attn_w = self.attn(self.norm1(x), return_attn=True)
            x = x + attn_out
            x = x + self.mlp(self.norm2(x))
            return x, attn_w
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

## 5. Full Vision Transformer

Assembles: patch embedding &rarr; prepend CLS token &rarr; add position embeddings &rarr; `depth` encoder blocks &rarr; take the CLS token's final vector &rarr; classification head. This mirrors the pipeline from the walkthrough end to end.

In [7]:
class VisionTransformer(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_channels=3, num_classes=10,
                 embed_dim=192, depth=6, num_heads=3, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))          # learnable CLS token
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))  # E_pos, learned
        self.pos_dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            EncoderBlock(embed_dim, num_heads, mlp_ratio, dropout) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)                        # W_head

        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x, return_attn=False):
        B = x.shape[0]
        x = self.patch_embed(x)                                  # (B, N, D)
        cls_tokens = self.cls_token.expand(B, -1, -1)             # (B, 1, D)
        x = torch.cat([cls_tokens, x], dim=1)                     # (B, N+1, D)  z0 = [z_class ; patch tokens]
        x = x + self.pos_embed                                    # + E_pos
        x = self.pos_dropout(x)

        attn_maps = []
        for block in self.blocks:
            if return_attn:
                x, attn_w = block(x, return_attn=True)
                attn_maps.append(attn_w)
            else:
                x = block(x)

        x = self.norm(x)
        cls_final = x[:, 0]                                       # y = z_L^(0), the CLS token's final vector
        logits = self.head(cls_final)                             # y_hat = softmax(y W_head + b), softmax via loss

        if return_attn:
            return logits, attn_maps
        return logits

## 6. Shape walkthrough (sanity check)

This should mirror the shape table from the walkthrough (scaled down to CIFAR-10's 32x32 images / 4x4 patches instead of ViT-Base's 224x224 / 16x16, so training stays fast on a free Colab GPU).

In [8]:
model = VisionTransformer(
    img_size=32, patch_size=4, in_channels=3, num_classes=10,
    embed_dim=192, depth=6, num_heads=3, mlp_ratio=4.0, dropout=0.1
).to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {num_params:,}")

dummy = torch.randn(4, 3, 32, 32).to(device)
logits, attn_maps = model(dummy, return_attn=True)
print("Input image:      ", dummy.shape)
print("Logits:            ", logits.shape)                 # (4, 10)
print("Num encoder layers:", len(attn_maps))
print("Attention map shape (per layer):", attn_maps[0].shape)  # (4, heads, 65, 65) -- 65 = 64 patches + CLS

Total parameters: 2,693,578
Input image:       torch.Size([4, 3, 32, 32])
Logits:             torch.Size([4, 10])
Num encoder layers: 6
Attention map shape (per layer): torch.Size([4, 3, 65, 65])


## 7. Training on CIFAR-10

A small end-to-end training loop. This ViT-Tiny-ish config (6 layers, 192-dim, 3 heads) trains in a few minutes per epoch on a free Colab T4 GPU. Don't expect CNN-beating accuracy from a few epochs — remember from the walkthrough that ViT has little built-in inductive bias, so it typically needs more data or augmentation than a CNN to reach the same accuracy on a small dataset like CIFAR-10.

In [ ]:
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader

train_transform = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
test_transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

train_set = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=train_transform)
test_set = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2, drop_last=True)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2)

classes = train_set.classes
print("Classes:", classes)

 20%|██        | 34.9M/170M [08:30<24:53, 90.8kB/s]

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
num_epochs = 10
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
criterion = nn.CrossEntropyLoss()


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

In [ ]:
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    scheduler.step()
    train_loss = running_loss / len(train_set)
    test_acc = evaluate(model, test_loader)
    print(f"Epoch {epoch+1}/{num_epochs} | train loss: {train_loss:.4f} | test accuracy: {test_acc:.4f}")

## 8. Visualizing attention on a real image

Just like the ear/face/sky example from the walkthrough — pick one test image, pick a layer, and see which patches the **CLS token** attends to most. This is the model's own version of "what is it looking at to make its decision."

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

mean = np.array([0.4914, 0.4822, 0.4465])
std = np.array([0.2470, 0.2435, 0.2616])

sample_img, sample_label = test_set[0]
model.eval()
with torch.no_grad():
    logits, attn_maps = model(sample_img.unsqueeze(0).to(device), return_attn=True)
pred = logits.argmax(dim=1).item()

# Attention from the CLS token (query index 0) to all patches, last layer, averaged over heads
last_layer_attn = attn_maps[-1][0]                 # (heads, N+1, N+1)
cls_attn = last_layer_attn[:, 0, 1:].mean(dim=0)    # average over heads -> (N,) attention to each patch

grid_size = int(math.sqrt(cls_attn.shape[0]))       # 8x8 for 32x32 image with patch_size=4
cls_attn_grid = cls_attn.reshape(grid_size, grid_size).cpu().numpy()

img_np = sample_img.permute(1, 2, 0).numpy() * std + mean
img_np = np.clip(img_np, 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img_np)
axes[0].set_title(f"True: {classes[sample_label]} | Pred: {classes[pred]}")
axes[0].axis("off")

axes[1].imshow(img_np)
axes[1].imshow(cls_attn_grid, cmap="jet", alpha=0.5, extent=(0, 32, 32, 0))
axes[1].set_title("CLS token attention (last layer, avg over heads)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 9. Scaling up to ViT-Base / next steps

This notebook uses a small ViT-Tiny-ish config so it trains quickly on CIFAR-10 in Colab. The real ViT-Base config from the original paper (and the shape table from the walkthrough) is:

| Config | img_size | patch_size | embed_dim | depth | num_heads | params |
|---|---|---|---|---|---|---|
| This notebook | 32 | 4 | 192 | 6 | 3 | ~2.7M |
| ViT-Base | 224 | 16 | 768 | 12 | 12 | ~86M |
| ViT-Large | 224 | 16 | 1024 | 24 | 16 | ~307M |

To scale up:
- Swap `img_size`/`patch_size`/`embed_dim`/`depth`/`num_heads` in the `VisionTransformer(...)` call.
- Real ViT is pretrained on very large datasets (ImageNet-21k or JFT-300M) before fine-tuning — from-scratch training on a small dataset like CIFAR-10 is exactly the regime where ViT's weak inductive bias hurts most, which is why this demo uses heavier data augmentation and won't match a CNN's accuracy from a few epochs.
- For a stronger from-scratch baseline on small datasets, look into DeiT-style training recipes (distillation from a CNN teacher, stronger augmentation such as RandAugment/Mixup/CutMix, and longer schedules).
- To load real pretrained ViT weights instead of training from scratch, the `timm` library (`pip install timm`) provides `timm.create_model('vit_base_patch16_224', pretrained=True)` with the same architecture implemented here.